# 12 · HOG+SVM test, rapor şekilleri, dağıtım paketi

Notebook 11 çalışmış olmalı (`outputs/final/` dolu). Önce GPU açın (HOG özniteliği CPU'da çıkar, yaklaşık 2-3 dk).

1. HOG+SVM taban çizgisi test kümesinde **tek sefer** değerlendirilir
2. Rapor şekilleri (Şekil 1-7): akış şeması, mimari, eğitim eğrileri, karışıklık matrisleri, ROC/PR, güvenilirlik, verimlilik
3. Dağıtım paketi (`deploy/`): özgün model + `predict.py` denemesi

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
# Kod ve çıktılar Drive'da tek klasörde durur; ham veri oturum diskinde (/content/data) tutulur.
PROJECT = next((p for p in ('/content/drive/MyDrive/YZProje', '/content/drive/MyDrive/YZProje/YZProje')
                if os.path.isfile(p + '/src/config.py')), None)
assert PROJECT, "Drive'da YZProje/src/config.py bulunamadı: proje klasörünü MyDrive/YZProje olarak yerleştirin."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = PROJECT + '/outputs'
os.environ['KNEE_DATA_ROOT'] = '/content/data'
print('Proje klasörü:', PROJECT)

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0 timm==1.0.30 scikit-image
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Görüntüler oturum diskinde yoksa: önce Drive'daki arşiv kopyası, o da yoksa bağlantıdan indirip arşivi Drive'a kaydeder.
from src import config as C, data_prep
print('Veri hazır:', data_prep.ensure_data())

## 1 · HOG+SVM test (tek sefer)

In [ ]:
!python -m src.baseline --final-test
from src import final_eval as F
F.baseline_summary().T

## 2 · Şekiller

In [ ]:
from src import figures_final as FF
print(FF.run_all())

In [ ]:
from IPython.display import Image, display
from src import config as C
for f in sorted(C.FIG_DIR.glob('fig0[1-7]*.png')):
    print(f.name); display(Image(str(f), width=800))

## 3 · Dağıtım paketi ve predict.py denemesi

In [ ]:
from src import export
print(export.export())

In [ ]:
from src.data import read_split
from src.data_prep import get_dataset_root
r = read_split('val').iloc[0]
!python predict.py "{get_dataset_root() / r.relpath}"
print('gerçek derece:', r.label)

## 4 · Çıktıları kaydet

In [ ]:
import shutil
from src import config as C
shutil.make_archive('/content/sekiller_ve_deploy', 'zip', str(C.FIG_DIR))
shutil.copy('/content/sekiller_ve_deploy.zip', str(C.OUTPUT_DIR / 'sekiller_ve_deploy.zip'))
print('hazır: outputs/sekiller_ve_deploy.zip')